In [ ]:
# =========================
# 🎯 CLUSTERING AVANCÉ - NEO4J NORTHWIND
# =========================

# =========================
# 📦 Librairies
# =========================
from neo4j import GraphDatabase
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# =========================
# 🔌 Connexion Neo4j
# =========================
uri = "bolt://localhost:7687"
user = "neo4j"
password = "adminadmin"
driver = GraphDatabase.driver(uri, auth=(user, password))

print("✅ Connexion établie avec Neo4j")

# =========================
# 📊 Récupération enrichie des données
# =========================

# D'abord, testons la connexion et la structure des données
print("\n🔍 Test de la structure de la base Neo4j...")

test_query = """
MATCH (c:Customer)
RETURN count(c) AS nb_customers
"""

with driver.session() as session:
    result = session.run(test_query)
    nb_customers = result.single()['nb_customers']
    print(f"   ✅ {nb_customers} clients trouvés dans la base")

# Requête adaptée selon la structure Northwind standard
query = """
MATCH (c:Customer)-[:PURCHASED]->(o:Order)
MATCH (o)-[orders:ORDERS]->(p:Product)
WITH c,
     COUNT(DISTINCT o) AS num_orders,
     SUM(toFloat(o.freight)) AS total_freight,
     AVG(toFloat(orders.quantity) * toFloat(p.unitPrice)) AS avg_order_value,
     SUM(toFloat(orders.quantity) * toFloat(p.unitPrice)) AS total_revenue,
     COUNT(DISTINCT p) AS num_different_products,
     AVG(toFloat(orders.quantity)) AS avg_quantity_per_order
RETURN id(c) AS customer_id,
       c.companyName AS company,
       num_orders,
       total_freight,
       avg_order_value,
       total_revenue,
       num_different_products,
       avg_quantity_per_order
ORDER BY customer_id

"""


with driver.session() as session:
    result = session.run(query)
    records = [dict(record) for record in result]
    df = pd.DataFrame(records)

# Vérification des données
if df.empty:
    print("\n❌ ERREUR : Aucune donnée récupérée!")
    print("   Vérifiez que :")
    print("   1. La base Northwind est bien chargée")
    print("   2. Les relations PURCHASED et ORDERS existent")
    print("   3. Neo4j est accessible")
    driver.close()
    raise ValueError("Impossible de récupérer les données depuis Neo4j")

print(f"\n📈 Données récupérées : {len(df)} clients")
print("\n🔍 Aperçu des données :")
print(df.head())

print("\n📊 Statistiques descriptives :")
print(df.describe())

print("\n📋 Info sur les colonnes :")
print(df.info())

# =========================
# 🧹 Préparation des données
# =========================
# Sélection des features pour le clustering
feature_cols = ['num_orders', 'total_freight', 'avg_order_value', 
                'total_revenue', 'num_different_products', 'avg_quantity_per_order']

X = df[feature_cols].fillna(0)

print("\n✨ Features sélectionnées :")
print(X.head())

# Normalisation (IMPORTANT!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n⚖️ Données normalisées avec StandardScaler")

# =========================
# 🔍 Méthode du coude (Elbow Method)
# =========================
print("\n🔍 Recherche du nombre optimal de clusters...")

inertias = []
silhouette_scores = []
K_range = range(2, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Méthode du coude
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Nombre de clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertie', fontsize=12)
axes[0].set_title('📉 Méthode du Coude (Elbow Method)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Score de silhouette
axes[1].plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Nombre de clusters (k)', fontsize=12)
axes[1].set_ylabel('Score de Silhouette', fontsize=12)
axes[1].set_title('🎯 Score de Silhouette', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Meilleur k selon silhouette
best_k = K_range[np.argmax(silhouette_scores)]
best_k = 3
print(f"\n✅ Nombre optimal de clusters (selon silhouette) : {best_k}")

# =========================
# 🎯 Clustering KMeans optimal
# =========================
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print(f"\n🎨 Clustering effectué avec {best_k} clusters")
print(f"📊 Score de silhouette final : {silhouette_score(X_scaled, df['cluster']):.3f}")

# =========================
# 📊 Profiling des clusters
# =========================
print("\n" + "="*60)
print("📋 PROFIL DES CLUSTERS")
print("="*60)

for cluster_id in sorted(df['cluster'].unique()):
    cluster_data = df[df['cluster'] == cluster_id]
    print(f"\n🏷️  CLUSTER {cluster_id}")
    print(f"   Taille: {len(cluster_data)} clients ({len(cluster_data)/len(df)*100:.1f}%)")
    print(f"   Commandes moyennes: {cluster_data['num_orders'].mean():.1f}")
    print(f"   Fret total moyen: {cluster_data['total_freight'].mean():.2f} €")
    print(f"   Revenu total moyen: {cluster_data['total_revenue'].mean():.2f} €")
    print(f"   Valeur commande moyenne: {cluster_data['avg_order_value'].mean():.2f} €")
    print(f"   Produits différents: {cluster_data['num_different_products'].mean():.1f}")
    print("   Top 3 clients:")
    for idx, row in cluster_data.nlargest(3, 'total_revenue').iterrows():
        print(f"      • {row['company']}: {row['total_revenue']:.2f} €")

# =========================
# 📊 Visualisations multiples
# =========================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Scatter plot: Commandes vs Fret
sns.scatterplot(data=df, x='num_orders', y='total_freight', 
                hue='cluster', palette='Set2', s=150, alpha=0.7, 
                edgecolor='black', linewidth=0.5, ax=axes[0,0])
axes[0,0].set_title('📦 Commandes vs Fret Total', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Nombre de commandes')
axes[0,0].set_ylabel('Total du fret (€)')

# 2. Scatter plot: Revenu vs Produits
sns.scatterplot(data=df, x='total_revenue', y='num_different_products', 
                hue='cluster', palette='Set2', s=150, alpha=0.7,
                edgecolor='black', linewidth=0.5, ax=axes[0,1])
axes[0,1].set_title('💰 Revenu vs Diversité Produits', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('Revenu total (€)')
axes[0,1].set_ylabel('Nombre de produits différents')

# 3. Distribution des clusters
cluster_counts = df['cluster'].value_counts().sort_index()
axes[1,0].bar(cluster_counts.index, cluster_counts.values, 
              color=sns.color_palette("Set2", best_k), edgecolor='black', linewidth=1.5)
axes[1,0].set_title('📊 Distribution des Clusters', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Cluster')
axes[1,0].set_ylabel('Nombre de clients')
axes[1,0].grid(axis='y', alpha=0.3)

# 4. Boxplot: Revenu par cluster
sns.boxplot(data=df, x='cluster', y='total_revenue', palette='Set2', ax=axes[1,1])
axes[1,1].set_title('💵 Distribution du Revenu par Cluster', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('Cluster')
axes[1,1].set_ylabel('Revenu total (€)')

plt.tight_layout()
plt.show()

# =========================
# 🔥 Heatmap des moyennes par cluster
# =========================
cluster_means = df.groupby('cluster')[feature_cols].mean()

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means.T, annot=True, fmt='.1f', cmap='YlOrRd', 
            linewidths=0.5, linecolor='gray', cbar_kws={'label': 'Valeur moyenne'})
plt.title('🔥 Heatmap des Caractéristiques Moyennes par Cluster', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Caractéristiques', fontsize=12)
plt.tight_layout()
plt.show()

# =========================
# 🚨 Détection d'anomalies
# =========================
print("\n🚨 DÉTECTION D'ANOMALIES")

iso_forest = IsolationForest(contamination=0.1, random_state=42)
df['anomaly'] = iso_forest.fit_predict(X_scaled)
df['anomaly_label'] = df['anomaly'].map({1: 'Normal', -1: 'Anomalie'})

n_anomalies = (df['anomaly'] == -1).sum()
print(f"   Anomalies détectées : {n_anomalies} clients ({n_anomalies/len(df)*100:.1f}%)")

# Visualisation des anomalies
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='num_orders', y='total_revenue', 
                hue='anomaly_label', palette={'Normal': 'lightblue', 'Anomalie': 'red'},
                s=150, alpha=0.7, edgecolor='black', linewidth=0.5)
plt.title('🚨 Détection d\'Anomalies (Isolation Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Nombre de commandes')
plt.ylabel('Revenu total (€)')
plt.legend(title='Statut', loc='best')
plt.tight_layout()
plt.show()

if n_anomalies > 0:
    print("\n🔍 Clients anomaliques détectés :")
    print(df[df['anomaly'] == -1][['company', 'num_orders', 'total_revenue', 'cluster']].to_string())

# =========================
# 💾 Réinjection dans Neo4j
# =========================
print("\n💾 Réinjection des clusters dans Neo4j...")

def update_cluster(tx, customer_id, cluster, anomaly):
    tx.run("""
        MATCH (c:Customer) 
        WHERE id(c) = $customer_id
        SET c.cluster = $cluster,
            c.is_anomaly = $anomaly
    """, customer_id=customer_id, cluster=int(cluster), anomaly=bool(anomaly == -1))

with driver.session() as session:
    for _, row in df.iterrows():
        session.execute_write(update_cluster, row['customer_id'], 
                            row['cluster'], row['anomaly'])

print("✅ Clusters et anomalies enregistrés dans Neo4j")

# =========================
# 🎨 DBSCAN (bonus)
# =========================
print("\n🎨 Clustering DBSCAN (détection basée sur la densité)...")

dbscan = DBSCAN(eps=0.5, min_samples=3)
df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

n_clusters_dbscan = len(set(df['dbscan_cluster'])) - (1 if -1 in df['dbscan_cluster'] else 0)
n_noise = list(df['dbscan_cluster']).count(-1)

print(f"   Clusters DBSCAN : {n_clusters_dbscan}")
print(f"   Points de bruit (outliers) : {n_noise}")

# =========================
# 📈 Résumé final
# =========================
print("\n" + "="*60)
print("🎉 ANALYSE TERMINÉE")
print("="*60)
print(f"✅ {len(df)} clients analysés")
print(f"✅ {best_k} clusters créés (KMeans)")
print(f"✅ {n_anomalies} anomalies détectées")
print("✅ Données sauvegardées dans Neo4j")
print("\n💡 Prochaines étapes suggérées :")
print("   1. Analyser les produits préférés par cluster")
print("   2. Créer des recommandations personnalisées")
print("   3. Identifier les clients à risque de churn")
print("   4. Visualiser le graph dans Neo4j Browser")
print("="*60)

# Fermeture de la connexion
driver.close()
print("\n🔌 Connexion Neo4j fermée")